# Prepare data

CSV trajectories from `00_make_dataset.ipynb` → `.npz` shards the model reads.

Two shards first, checked, before committing to all 100 (~2–5 min, ~2.3GB).
Conversion lives in `dataset.py`; this notebook only drives it.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np

from config import Config
from dataset import build_dataloader, check_shards, convert_all

CSV_DIR = "data/square_room_100steps_2.2m_1000000"
SHARD_DIR = "data/shards"
cfg = Config()
print(ROOT)

## Two shards, as a trial run

In [ ]:
paths = convert_all(CSV_DIR, SHARD_DIR, shard_indices=[0, 1])
check_shards(paths)
print("\nshapes/dtypes/finiteness all check out")

with np.load(paths[0]) as npz:
    for k in npz.files:
        print(f"  {k:12} {str(npz[k].shape):18} {npz[k].dtype}")

In [ ]:
# What the model will actually receive.
loader = build_dataloader(SHARD_DIR, shard_indices=[0, 1],
                          batch_size=cfg.train.minibatch_size)
assert len(loader.dataset) == 20_000, len(loader.dataset)

batch = next(iter(loader))
b = cfg.train.minibatch_size
assert batch["init_pos"].shape == (b, 2)
assert batch["init_hd"].shape == (b, 1)
assert batch["ego_vel"].shape == (b, 100, 3)
assert batch["target_pos"].shape == (b, 100, 2)
assert batch["target_hd"].shape == (b, 100, 1)
print("batch:", {k: tuple(v.shape) for k, v in batch.items()})

In [ ]:
# Sanity on the values themselves: positions centred on the arena, ego_vel's
# last two components a unit vector (sin, cos of the same angle).
pos = batch["target_pos"].numpy()
ego = batch["ego_vel"].numpy()
half = cfg.task.env_size / 2
print(f"target_pos range  x [{pos[..., 0].min():+.2f}, {pos[..., 0].max():+.2f}] "
      f"y [{pos[..., 1].min():+.2f}, {pos[..., 1].max():+.2f}]   (arena +-{half})")
print(f"speed             mean {ego[..., 0].mean():.3f} m/s, max {ego[..., 0].max():.3f}")
print(f"sin^2+cos^2       {np.abs(ego[..., 1]**2 + ego[..., 2]**2 - 1).max():.2e} off 1")

## All 100 shards

Only once the above passes. A few minutes, and `data/shards/` is gitignored.

In [ ]:
all_paths = convert_all(CSV_DIR, SHARD_DIR, shard_indices=None, verbose=False)
check_shards(all_paths)
total = sum(os.path.getsize(p) for p in all_paths) / 1e9
print(f"{len(all_paths)} shards, {total:.2f}GB, all checks passed")